In [28]:
# ============================================================
# CELL 0 — Environment Setup (run this first, every session)
# ============================================================
# USE_DRIVE = True  → Google Colab + Google Drive
# USE_DRIVE = False → Local machine  ← DEFAULT
# ============================================================

import os, sys
from pathlib import Path

USE_DRIVE = False  # ← change to True only on Colab + Drive

# ── Auto-detect Colab ────────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Resolve BASE (repo root) ─────────────────────────────────────────────────
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/retail-demand-forecasting')
elif IN_COLAB:
    BASE = Path('/content/dl-assignment')
else:
    # Local: CWD is either repo root or notebooks/ — handle both
    _cwd = Path(os.getcwd())
    BASE = _cwd.parent if _cwd.name == 'notebooks' else _cwd

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_PATH      = BASE / 'data' / 'm5' / 'extracted'
PROCESSED_PATH = BASE / 'data' / 'm5' / 'processed'
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

CALENDAR_PATH = DATA_PATH / 'calendar.csv'
PRICES_PATH   = DATA_PATH / 'sell_prices.csv'
SALES_PATH    = DATA_PATH / 'sales_train_validation.csv'

# ── Add src/ to Python path ───────────────────────────────────────────────────
SRC_PATH = str(BASE / 'src')
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# ── GPU check ─────────────────────────────────────────────────────────────────
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Environment   : {"Google Colab" if IN_COLAB else "Local"}')
print(f'USE_DRIVE     : {USE_DRIVE}')
print(f'BASE          : {BASE}')
print(f'Device        : {device}')
if torch.cuda.is_available():
    print(f'GPU           : {torch.cuda.get_device_name(0)}')
print(f'DATA_PATH     : {DATA_PATH}')
print(f'DATA exists?  : {DATA_PATH.exists()}')
if not DATA_PATH.exists():
    print()
    print('[WARNING] DATA_PATH not found! Expected M5 CSVs in:')
    print(f'          {DATA_PATH}')


Environment   : Local
USE_DRIVE     : False
BASE          : e:\dl-assignment
Device        : cuda
GPU           : NVIDIA GeForce RTX 3050 6GB Laptop GPU
DATA_PATH     : e:\dl-assignment\data\m5\extracted
DATA exists?  : True


In [29]:
# Create notebooks output folder (cross-platform)
nb_out = Path('/content/notebooks') if IN_COLAB else BASE / 'notebooks'
nb_out.mkdir(parents=True, exist_ok=True)
print('Notebook directory ready:', nb_out)


Notebook directory ready: e:\dl-assignment\notebooks


In [30]:
import os
import pandas as pd
import numpy as np

from pathlib import Path

print("Libraries loaded successfully")

Libraries loaded successfully


In [31]:
calendar = pd.read_csv(CALENDAR_PATH)
prices   = pd.read_csv(PRICES_PATH)
sales    = pd.read_csv(SALES_PATH)

print("Calendar shape:", calendar.shape)
print("Prices shape:", prices.shape)
print("Sales shape:", sales.shape)

Calendar shape: (1969, 14)
Prices shape: (6841121, 4)
Sales shape: (30490, 1919)


In [32]:
calendar_clean = calendar.copy()
prices_clean   = prices.copy()
sales_clean    = sales.copy()

print("Working copies created.")

Working copies created.


In [33]:
print("===== DUPLICATE ROW CHECK =====")
print("Calendar duplicates:", calendar_clean.duplicated().sum())
print("Prices duplicates:",   prices_clean.duplicated().sum())
print("Sales duplicates:",    sales_clean.duplicated().sum())

===== DUPLICATE ROW CHECK =====
Calendar duplicates: 0
Prices duplicates: 0
Sales duplicates: 0


In [34]:
series_id_columns = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

duplicate_series = sales_clean.duplicated(
    subset=series_id_columns
).sum()

print(
    "Duplicate product-store series:",
    duplicate_series
)

Duplicate product-store series: 0


In [35]:
print(calendar_clean["date"].dtype)

calendar_clean["date"] = pd.to_datetime(calendar_clean["date"], errors="coerce")

print(calendar_clean["date"].dtype)

str
datetime64[us]


In [36]:
invalid_dates = calendar_clean["date"].isna().sum()
print("Invalid dates after conversion:", invalid_dates)

Invalid dates after conversion: 0


In [37]:
calendar_clean = calendar_clean.sort_values("date").reset_index(drop=True)

print("Earliest date:", calendar_clean["date"].min())
print("Latest date:",   calendar_clean["date"].max())

Earliest date: 2011-01-29 00:00:00
Latest date: 2016-06-19 00:00:00


In [38]:
sales_columns = [c for c in sales_clean.columns if c.startswith("d_")]
print("Number of daily sales columns:", len(sales_columns))

sales_clean[sales_columns] = sales_clean[sales_columns].apply(pd.to_numeric, errors="coerce")

print(sales_clean[sales_columns].dtypes.value_counts())

Number of daily sales columns: 1913
int64    1913
Name: count, dtype: int64


In [39]:
negative_sales_count = (sales_clean[sales_columns] < 0).sum().sum()
print("Negative demand observations:", negative_sales_count)

Negative demand observations: 0


In [40]:
missing_sales_count = sales_clean[sales_columns].isna().sum().sum()
print("Missing daily demand observations:", missing_sales_count)

Missing daily demand observations: 0


In [41]:
identifier_columns = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
print(sales_clean[identifier_columns].isna().sum())

item_id     0
dept_id     0
cat_id      0
store_id    0
state_id    0
dtype: int64


In [42]:
print("Price dtypes:")
print(prices_clean.dtypes)

prices_clean["sell_price"] = pd.to_numeric(prices_clean["sell_price"], errors="coerce")

print("Missing sell prices:", prices_clean["sell_price"].isna().sum())
print("Negative sell prices:", (prices_clean["sell_price"] < 0).sum())

Price dtypes:
store_id          str
item_id           str
wm_yr_wk        int64
sell_price    float64
dtype: object
Missing sell prices: 0
Negative sell prices: 0


In [43]:
prices_clean = (
    prices_clean
    .sort_values(["store_id", "item_id", "wm_yr_wk"])
    .reset_index(drop=True)
)

prices_clean.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,FOODS_1_001,11101,2.0
1,CA_1,FOODS_1_001,11102,2.0
2,CA_1,FOODS_1_001,11103,2.0
3,CA_1,FOODS_1_001,11104,2.0
4,CA_1,FOODS_1_001,11105,2.0


In [44]:
print("First day IDs:", calendar_clean["d"].head().tolist())
print("Last day IDs:",  calendar_clean["d"].tail().tolist())

print("Duplicate calendar day IDs:", calendar_clean["d"].duplicated().sum())

First day IDs: ['d_1', 'd_2', 'd_3', 'd_4', 'd_5']
Last day IDs: ['d_1965', 'd_1966', 'd_1967', 'd_1968', 'd_1969']
Duplicate calendar day IDs: 0


In [45]:
# PROCESSED_PATH already defined in Cell 0 above
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

calendar_clean.to_csv(PROCESSED_PATH / "calendar_clean.csv", index=False)
prices_clean.to_csv(PROCESSED_PATH / "sell_prices_clean.csv", index=False)
sales_clean.to_csv(PROCESSED_PATH / "sales_train_validation_clean.csv", index=False)

print("Cleaned datasets saved.")

Cleaned datasets saved.


In [46]:
print(list(PROCESSED_PATH.iterdir()))

[WindowsPath('e:/dl-assignment/data/m5/processed/calendar_clean.csv'), WindowsPath('e:/dl-assignment/data/m5/processed/sales_train_validation_clean.csv'), WindowsPath('e:/dl-assignment/data/m5/processed/sell_prices_clean.csv')]


In [47]:
# On Colab: copy preprocessing.py from repo src/ to /content/src/ if needed
# On local: src/ is already in sys.path from Cell 0 — nothing to do
import shutil

src_file = BASE / 'src' / 'preprocessing.py'

if IN_COLAB and not (Path('/content/src') / 'preprocessing.py').exists():
    colab_src = Path('/content/src')
    colab_src.mkdir(parents=True, exist_ok=True)
    if src_file.exists():
        shutil.copy(src_file, colab_src / 'preprocessing.py')
        print("Copied preprocessing.py to /content/src/")
    else:
        print("[WARNING] preprocessing.py not found at:", src_file)
else:
    print("preprocessing.py available at:", SRC_PATH)


preprocessing.py available at: e:\dl-assignment\src


In [48]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nPython import paths containing src:")
for path in sys.path:
    if 'src' in path.lower():
        print("-", path)

print("\nFiles in SRC_PATH:")
src_dir = Path(SRC_PATH)
if src_dir.exists():
    print(os.listdir(str(src_dir)))
else:
    print("[WARNING] SRC_PATH not found:", src_dir)

print("\npreprocessing.py exists:", (src_dir / 'preprocessing.py').exists())


Current working directory:
e:\dl-assignment\notebooks

Python import paths containing src:
- e:\dl-assignment\src

Files in SRC_PATH:
['preprocessing.py', '__pycache__']

preprocessing.py exists: True


In [49]:
import importlib

# SRC_PATH already inserted in Cell 0; reload in case of re-run
importlib.invalidate_caches()

import preprocessing

print("preprocessing module loaded successfully")
print("Module location:", preprocessing.__file__)


preprocessing module loaded successfully
Module location: e:\dl-assignment\src\preprocessing.py


In [50]:
from preprocessing import (
    clean_calendar,
    clean_prices,
    clean_sales
)

print("All cleaning functions imported successfully")

All cleaning functions imported successfully


In [51]:
calendar_clean = clean_calendar(calendar)
prices_clean = clean_prices(prices)
sales_clean = clean_sales(sales)

print("Cleaning functions executed successfully.")
print()
print("Original vs cleaned shapes:")
print("Calendar:", calendar.shape, "→", calendar_clean.shape)
print("Prices:  ", prices.shape, "→", prices_clean.shape)
print("Sales:   ", sales.shape, "→", sales_clean.shape)

Cleaning functions executed successfully.

Original vs cleaned shapes:
Calendar: (1969, 14) → (1969, 14)
Prices:   (6841121, 4) → (6841121, 4)
Sales:    (30490, 1919) → (30490, 1919)


In [52]:
print("===== CLEANING VALIDATION =====")

print("Calendar shape:", calendar_clean.shape)
print("Prices shape:", prices_clean.shape)
print("Sales shape:", sales_clean.shape)

print("\nCalendar duplicates:",
      calendar_clean.duplicated().sum())

print("Price duplicates:",
      prices_clean.duplicated().sum())

print("Sales duplicates:",
      sales_clean.duplicated().sum())

print("\nMissing sales values:",
      sales_clean[sales_columns].isna().sum().sum())

print("Negative sales values:",
      (sales_clean[sales_columns] < 0).sum().sum())

print("\nInvalid calendar dates:",
      calendar_clean["date"].isna().sum())

print("\nCleaning validation completed.")

===== CLEANING VALIDATION =====
Calendar shape: (1969, 14)
Prices shape: (6841121, 4)
Sales shape: (30490, 1919)

Calendar duplicates: 0
Price duplicates: 0
Sales duplicates: 0

Missing sales values: 0
Negative sales values: 0

Invalid calendar dates: 0

Cleaning validation completed.


In [53]:
# PROCESSED_PATH is already defined in Cell 0 — no need to redefine
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

calendar_clean.to_csv(
    PROCESSED_PATH / 'calendar_clean.csv',
    index=False
)

prices_clean.to_csv(
    PROCESSED_PATH / 'sell_prices_clean.csv',
    index=False
)

sales_clean.to_csv(
    PROCESSED_PATH / 'sales_train_validation_clean.csv',
    index=False
)

print("Cleaned datasets saved successfully.")
print()
print("Saved files:")

for file in PROCESSED_PATH.iterdir():
    print("-", file.name)


Cleaned datasets saved successfully.

Saved files:
- calendar_clean.csv
- sales_train_validation_clean.csv
- sell_prices_clean.csv


In [54]:
print("===== SAVED FILE VALIDATION =====")

calendar_check = pd.read_csv(
    PROCESSED_PATH / "calendar_clean.csv"
)

prices_check = pd.read_csv(
    PROCESSED_PATH / "sell_prices_clean.csv"
)

sales_check = pd.read_csv(
    PROCESSED_PATH / "sales_train_validation_clean.csv"
)

print("Calendar:", calendar_check.shape)
print("Prices:  ", prices_check.shape)
print("Sales:   ", sales_check.shape)

print("\nSaved files loaded successfully.")

===== SAVED FILE VALIDATION =====
Calendar: (1969, 14)
Prices:   (6841121, 4)
Sales:    (30490, 1919)

Saved files loaded successfully.


# Missing-Value & Data-Quality Preprocessing

The goal here is not to blindly fill every missing value. For the M5 dataset, some missing values—especially calendar event fields—can represent “no event”, so we need to distinguish meaningful missingness from actual data-quality problems.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries loaded successfully")

## Confirm the cleaned datasets

In [ ]:
print("===== CLEANED DATASETS =====")

print("Calendar:", calendar_clean.shape)
print("Prices:  ", prices_clean.shape)
print("Sales:   ", sales_clean.shape)

## Analyze missing values

Now check missing values across all three datasets.

In [ ]:
print("===== MISSING VALUE ANALYSIS =====")

calendar_missing = calendar_clean.isna().sum()
prices_missing = prices_clean.isna().sum()
sales_missing = sales_clean.isna().sum()

print("\n--- Calendar ---")
print(calendar_missing[calendar_missing > 0])

print("\n--- Prices ---")
print(prices_missing[prices_missing > 0])

print("\n--- Sales ---")
print(sales_missing[sales_missing > 0])

## Calculate missing-value percentages

Instead of looking only at counts, calculate percentages.

In [ ]:
def missing_summary(df, name):
    summary = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percentage": (
            df.isna().mean() * 100
        )
    })

    summary = summary[
        summary["missing_count"] > 0
    ]

    print(f"\n===== {name} =====")

    if summary.empty:
        print("No missing values found.")
    else:
        print(summary.sort_values(
            "missing_percentage",
            ascending=False
        ))


missing_summary(calendar_clean, "CALENDAR")
missing_summary(prices_clean, "PRICES")
missing_summary(sales_clean, "SALES")

## Inspect calendar missing values

The calendar contains event-related fields.

In [ ]:
calendar_missing_columns = [
    col for col in calendar_clean.columns
    if calendar_clean[col].isna().sum() > 0
]

print("Calendar columns containing missing values:")
print(calendar_missing_columns)

for col in calendar_missing_columns:
    print(f"\n--- {col} ---")
    print("Missing:", calendar_clean[col].isna().sum())
    print(
        "Non-missing:",
        calendar_clean[col].notna().sum()
    )

## Understand whether calendar missing values are meaningful

For event fields, inspect the actual values.

In [ ]:
for col in ["event_name_1", "event_type_1",
            "event_name_2", "event_type_2"]:

    if col in calendar_clean.columns:
        print(f"\n===== {col} =====")
        print(
            calendar_clean[col]
            .value_counts(dropna=False)
            .head(15)
        )

### Important decision

For these event columns, `NaN` does **not automatically mean bad data**.

It generally represents:

> No event was recorded for that calendar day.

We therefore should **not delete those rows**.

## Create explicit "no event" values

For categorical event fields, replacing missing values with `"None"` makes the information explicit and easier for downstream feature engineering.

In [ ]:
event_columns = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

for col in event_columns:
    if col in calendar_clean.columns:
        calendar_clean[col] = (
            calendar_clean[col]
            .fillna("None")
        )

print("Calendar event missing values handled.")

## Verify event columns

In [ ]:
print("===== EVENT COLUMN VALIDATION =====")

for col in event_columns:
    if col in calendar_clean.columns:
        print(
            f"{col}:",
            calendar_clean[col].isna().sum(),
            "missing values"
        )

## Check remaining calendar missing values

Now check all remaining missing values.

In [ ]:
print("===== REMAINING CALENDAR MISSING VALUES =====")

remaining_calendar_missing = (
    calendar_clean.isna().sum()
)

print(
    remaining_calendar_missing[
        remaining_calendar_missing > 0
    ]
)

## Check price missing values

Now examine the price dataset separately.

In [ ]:
print("===== PRICE DATA QUALITY =====")

print("Missing sell prices:",
      prices_clean["sell_price"].isna().sum())

print("Negative sell prices:",
      (prices_clean["sell_price"] < 0).sum())

print("Zero sell prices:",
      (prices_clean["sell_price"] == 0).sum())

## Check sales missing values

We already established there were zero missing daily sales, but let's explicitly document it.

In [ ]:
sales_columns = [
    col for col in sales_clean.columns
    if col.startswith("d_")
]

missing_sales = (
    sales_clean[sales_columns]
    .isna()
    .sum()
    .sum()
)

print("Number of daily sales columns:", len(sales_columns))
print("Missing daily sales values:", missing_sales)

## Check identifier columns

Make sure the product/store hierarchy has no missing identifiers.

In [ ]:
identifier_columns = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

print("===== IDENTIFIER MISSING VALUES =====")

for col in identifier_columns:
    print(
        f"{col}:",
        sales_clean[col].isna().sum()
    )

## Check categorical consistency

Now we'll check whether the hierarchical identifiers contain unexpected values.

In [ ]:
print("===== CATEGORICAL VALUE COUNTS =====")

for col in identifier_columns:
    print(f"\n{col}")
    print(
        sales_clean[col]
        .value_counts()
        .head(20)
    )

## Check for invalid sales values again

Because sales are our target, perform a final quality check.

In [ ]:
print("===== SALES VALUE VALIDATION =====")

negative_count = (
    sales_clean[sales_columns] < 0
).sum().sum()

missing_count = (
    sales_clean[sales_columns].isna()
).sum().sum()

print("Negative sales values:", negative_count)
print("Missing sales values:", missing_count)

## Reload the updated module

Because Python already imported the old version, reload it.

In [ ]:
import sys
import importlib
from pathlib import Path

# Ensure SRC_PATH is loaded correctly for both colab and local
if 'SRC_PATH' in locals() and SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)
elif "/content/src" not in sys.path and Path("/content/src").exists():
    sys.path.insert(0, "/content/src")

import preprocessing
importlib.reload(preprocessing)

print("Updated preprocessing module loaded.")

## Import the new functions

In [ ]:
from preprocessing import (
    clean_calendar,
    clean_prices,
    clean_sales,
    handle_calendar_missing_values,
    missing_value_summary
)

print("All preprocessing functions imported successfully.")

## Apply the missing-value function

Now create the final calendar dataset:

In [ ]:
calendar_clean = handle_calendar_missing_values(
    calendar_clean
)

print("Calendar missing-value handling completed.")

## Final missing-value validation

In [ ]:
print("===== FINAL MISSING VALUE VALIDATION =====")

print("\nCalendar:")
print(
    calendar_clean.isna().sum()[
        calendar_clean.isna().sum() > 0
    ]
)

print("\nPrices:")
print(
    prices_clean.isna().sum()[
        prices_clean.isna().sum() > 0
    ]
)

print("\nSales:")
print(
    sales_clean.isna().sum()[
        sales_clean.isna().sum() > 0
    ]
)

## Save the updated cleaned calendar

Since we changed the calendar event columns, overwrite the processed calendar file:

In [ ]:
calendar_clean.to_csv(
    PROCESSED_PATH / "calendar_clean.csv",
    index=False
)

print("Updated calendar_clean.csv saved successfully.")

## Final Validation

In [ ]:
print("========================================")
print("             VALIDATION                 ")
print("========================================")

print("\nDataset shapes:")
print("Calendar:", calendar_clean.shape)
print("Prices:  ", prices_clean.shape)
print("Sales:   ", sales_clean.shape)

print("\nData quality:")
print(
    "Calendar missing:",
    calendar_clean.isna().sum().sum()
)

print(
    "Prices missing:",
    prices_clean.isna().sum().sum()
)

print(
    "Sales missing:",
    sales_clean.isna().sum().sum()
)

print(
    "Negative sales:",
    (sales_clean[sales_columns] < 0).sum().sum()
)

print("\nValidation completed.")